# T19 — ¿Bajamos la dificultad de este jefe?

Hackatón IIA · UADE · Pista: **Reglas**.

El sistema estima si un jugador abandonaría el juego si la dificultad del jefe no se reduce. **SÍ = 1**, **NO = 0**.

El umbral se define **solo con casos 1–14**. Los casos **15–20** quedan para evaluación. Se informa también una línea base trivial y las cuatro métricas obligatorias.

## 1. Datos originales

Se usa el CSV `T19.csv` del repositorio de la cátedra. El programa intenta abrirlo desde GitHub y, si falla, lee el archivo local `T19.csv` (se puede subir con el panel de archivos de Colab).

In [ ]:
import pandas as pd
from pathlib import Path

URL = "https://raw.githubusercontent.com/dharioprodigio/hackaton-iia-uade-2026/main/datos/T19.csv"
try:
    datos = pd.read_csv(URL)
except Exception as error:
    print("No se pudo descargar el CSV; se intentará leer T19.csv local.", error)
    datos = pd.read_csv("T19.csv")

assert len(datos) == 20, "Se esperaban exactamente 20 casos."
assert datos['caso'].tolist() == list(range(1, 21)), "Los casos deben estar ordenados 1 a 20."
assert (datos.loc[datos.caso <= 14, 'conjunto'] == 'entrenamiento').all()
assert (datos.loc[datos.caso >= 15, 'conjunto'] == 'prueba').all()

muertes_col = "Muertes seguidas (contra el jefe)"
vida_col = "Vida que le dejó al jefe (% en el mejor intento)"
horas_col = "Horas jugadas (en total)"
y_col = "resultado_real"

entrenamiento = datos[datos['conjunto'] == 'entrenamiento'].copy()
prueba = datos[datos['conjunto'] == 'prueba'].copy()
print('Casos entrenamiento:',len(entrenamiento),'| Casos prueba:',len(prueba))
print('Clases de entrenamiento:',entrenamiento[y_col].value_counts().sort_index().to_dict())
# Mostramos solo el conjunto de entrenamiento durante la construcción.
display(entrenamiento)

## 2. Construcción de la regla CON ENTRENAMIENTO

Entre los 14 ejemplos de entrenamiento, el máximo de horas en la clase SÍ es 43 y el mínimo de horas en la clase NO es 47. Elegimos su punto medio (45).

No usamos las etiquetas de los casos de prueba para definir el umbral.

**Cuidado:** que una sola característica separe estos pocos casos no demuestra que sea la causa del abandono. Es una regla didáctica, no un detector validado para videojuegos reales.

In [ ]:
# Estos extremos y el umbral se calculan SOLO con los 14 registros de entrenamiento.
max_horas_si = entrenamiento.loc[entrenamiento[y_col] == 1, horas_col].max()
min_horas_no = entrenamiento.loc[entrenamiento[y_col] == 0, horas_col].min()
umbral = (max_horas_si + min_horas_no) / 2

print(f'Máximo de horas en SÍ (entrenamiento): {max_horas_si}')
print(f'Mínimo de horas en NO (entrenamiento): {min_horas_no}')
print(f'Umbral obtenido con entrenamiento: {umbral} horas')

def decidir(muertes: float, vida_restante: float, horas: float) -> int:
    """SÍ (1): predecimos abandono si no se reduce dificultad; NO (0): mantener."""
    if muertes < 0 or horas < 0 or not (0 <= vida_restante <= 100):
        raise ValueError('Muertes y horas >= 0; vida restante entre 0 y 100.')
    return int(horas <= umbral)

print('Ejemplo A:', decidir(12, 55, 35), '(esperado 1)')
print('Ejemplo B:', decidir(17, 50, 92), '(esperado 0)')

## 3. Medición en ambos conjuntos

Calculamos **VP, FN, FP y VN** sin librerías adicionales, para que los números se puedan revisar. Después calculamos exactitud, precisión, sensibilidad y especificidad. Si un denominador fuese 0, la métrica se reportaría como «no definida».

In [ ]:
def evaluar(nombre, conjunto):
    reales = conjunto[y_col].astype(int).tolist()
    predichos = [decidir(fila[muertes_col], fila[vida_col], fila[horas_col])
                 for _, fila in conjunto.iterrows()]
    VP = sum(r == 1 and p == 1 for r, p in zip(reales, predichos))
    FN = sum(r == 1 and p == 0 for r, p in zip(reales, predichos))
    FP = sum(r == 0 and p == 1 for r, p in zip(reales, predichos))
    VN = sum(r == 0 and p == 0 for r, p in zip(reales, predichos))
    total = VP + FN + FP + VN
    assert total == len(conjunto), 'La matriz no suma el número de casos.'
    def div(a, b):
        return a / b if b else float('nan')
    print(f'\n{nombre.upper()} — {len(conjunto)} casos')
    matriz = pd.DataFrame([[VP, FN], [FP, VN]],
                          index=['Real SÍ', 'Real NO'], columns=['Predice SÍ', 'Predice NO'])
    display(matriz)
    metricas = {
        'Exactitud': div(VP + VN, total),
        'Precisión': div(VP, VP + FP),
        'Sensibilidad': div(VP, VP + FN),
        'Especificidad': div(VN, VN + FP),
    }
    for nombre_m, valor in metricas.items():
        print(f'  {nombre_m}: {valor:.1%}' if pd.notna(valor) else f'  {nombre_m}: no definida')
    detalle = conjunto[['caso', y_col]].copy()
    detalle['prediccion'] = predichos
    detalle['acierto'] = detalle[y_col] == detalle['prediccion']
    display(detalle)
    return {'VP':VP,'FN':FN,'FP':FP,'VN':VN,**metricas}

resultado_train = evaluar('Entrenamiento', entrenamiento)
resultado_test = evaluar('Prueba', prueba)

resumen = pd.DataFrame({'Entrenamiento':resultado_train,'Prueba':resultado_test})
print('RESUMEN:')
display(resumen)
assert (resultado_train['VP'],resultado_train['FN'],resultado_train['FP'],resultado_train['VN']) == (7,0,0,7)
assert (resultado_test['VP'],resultado_test['FN'],resultado_test['FP'],resultado_test['VN']) == (2,2,0,2)

## 4. Línea base trivial

Comparación con un sistema que **responde siempre NO** (no reduce nunca la dificultad). Se incluye también siempre SÍ para mostrar que la exactitud sola puede engañar.

In [ ]:
for nombre, conjunto in [('Entrenamiento', entrenamiento), ('Prueba', prueba)]:
    reales = conjunto[y_col].astype(int)
    pred_modelo = [decidir(fila[muertes_col], fila[vida_col], fila[horas_col]) for _,fila in conjunto.iterrows()]
    for sistema, pred in [('Regla T19',pd.Series(pred_modelo,index=conjunto.index)),
                          ('Siempre NO',pd.Series(0,index=conjunto.index)),
                          ('Siempre SÍ',pd.Series(1,index=conjunto.index))]:
        aciertos = int((reales == pred).sum())
        vp = int(((reales == 1) & (pred == 1)).sum())
        fn = int(((reales == 1) & (pred == 0)).sum())
        print(f'{nombre:13s} | {sistema:10s} | aciertos {aciertos}/{len(conjunto)} | exactitud {aciertos/len(conjunto):.1%} | VP {vp} | FN {fn}')

## 5. Caso nuevo en vivo

Modificá **solo los tres números** de la celda siguiente con los valores que te dé el profesor y ejecutala. No necesitás conocer el resultado real del jugador.

**Advertencia:** esta regla inicial solo utiliza las horas jugadas. Muertes y vida restante se reciben para mostrar el formato completo del caso, pero no cambian su decisión.

In [ ]:
# EDITÁ ESTOS VALORES con los datos del caso nuevo del profesor:
muertes_nuevo = 12
vida_restante_nuevo = 55  # 0 a 100: cuánto le quedó al jefe
horas_jugadas_nuevo = 35

prediccion = decidir(muertes_nuevo, vida_restante_nuevo, horas_jugadas_nuevo)
print('Respuesta:', 'SÍ (1)' if prediccion else 'NO (0)')
print('Acción recomendada:', 'BAJAR dificultad' if prediccion else 'MANTENER dificultad')
print(f'Motivo: horas jugadas ({horas_jugadas_nuevo}) '+
      ('<= ' if prediccion else '> ') + f'umbral ({umbral:.0f}).')